# FedEduAI — Experiment 3 (hybrid Know-Evo): the large-language-model channel, v21

This notebook implements only the LLM channel of the hybrid Algorithm 3: it grades every student answer of the Mohler corpus (2,442 answers, 85 questions) against the instructor's reference answer on a 0–5 scale with an open model (Qwen2.5-7B-Instruct, 4-bit quantisation) on a free T4 GPU.

The text-mining channel (TF-IDF/ROUGE/Jaccard with Ridge calibration) is computed separately; the hybrid score (mean of the two channels) is then evaluated against the mean of the two human graders (RMSE, Pearson, Spearman, accuracy within ±0.5 and ±1) in `FedEduAI_Validation_v13.ipynb`.

Execution: Runtime → Change runtime type → T4 GPU, then run the cells in order (40–70 minutes). Outputs are saved to `MyDrive/iudb/results/exp3/` as `llm_scores_v21_<model>.csv` and a zip archive.


In [1]:
# 1) Installation and Drive mount
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes pyarrow
from google.colab import drive; drive.mount('/content/drive')
import os, json, time, re, hashlib, numpy as np, pandas as pd, torch
OUT='/content/drive/MyDrive/iudb/results/exp3'; os.makedirs(OUT, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change the runtime type to GPU')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
Mounted at /content/drive
GPU: Tesla T4


In [2]:
# 2) Data: Mohler corpus (cleaned nkazi/MohlerASAG release); assignments E11/E12 rescaled from 0–10 to 0–5 (same preprocessing as the TM channel)
B='https://huggingface.co/datasets/nkazi/MohlerASAG/resolve/main/data'
df=pd.concat([pd.read_parquet(f'{B}/cleaned-oe-00001.parquet'), pd.read_parquet(f'{B}/cleaned-ce-00001.parquet')], ignore_index=True)
df['asg']=df.id.str.split('.').str[0]
mx=df.groupby('asg')[['score_grader_1','score_grader_2']].max().max(axis=1); s10=set(mx[mx>5].index)
df['scale10']=df.asg.isin(s10); df['g1']=np.where(df.scale10, df.score_grader_1/2, df.score_grader_1); df['g2']=np.where(df.scale10, df.score_grader_2/2, df.score_grader_2)
df=df[['id','asg','question','instructor_answer','student_answer','g1','g2','score_avg']]
sha=hashlib.sha256(pd.util.hash_pandas_object(df[['id','question','instructor_answer','student_answer','score_avg']], index=False).values.tobytes()).hexdigest()[:16]
print(len(df),'answers;', df.question.nunique(),'questions; rescaled assignments:', sorted(s10), '; data hash', sha)

2442 answers; 85 questions; rescaled assignments: ['E11', 'E12'] ; data hash 86b6af7101c1b3de


In [4]:
# 3) Model loading (4-bit NF4). If memory runs out, use Qwen/Qwen2.5-3B-Instruct
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL='Qwen/Qwen2.5-7B-Instruct'
tok=AutoTokenizer.from_pretrained(MODEL); tok.padding_side='left'
model=AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16), device_map='auto')
model.eval(); print('loaded', MODEL)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-7B-Instruct


In [5]:
# 4) Prompt: strict 0–5 grading (half points allowed) against the reference answer; the model returns a number only
SYSTEM=("You are an experienced computer-science instructor grading short-answer questions. "
        "Grade the STUDENT ANSWER against the REFERENCE ANSWER on a 0-5 scale (half points allowed): "
        "5 = fully correct and complete; 4 = correct with a minor omission; 3 = partially correct; 2 = mostly incorrect but shows some understanding; "
        "1 = largely incorrect; 0 = wrong, empty or irrelevant. Judge meaning, not wording. Output ONLY the number.")
def build(q, ref, ans):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':f"QUESTION: {q}\nREFERENCE ANSWER: {ref}\nSTUDENT ANSWER: {ans}\nScore (0-5):"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
prompts=[build(q,r,a) for q,r,a in zip(df.question, df.instructor_answer, df.student_answer)]
print(prompts[0][-600:])

shows some understanding; 1 = largely incorrect; 0 = wrong, empty or irrelevant. Judge meaning, not wording. Output ONLY the number.<|im_end|>
<|im_start|>user
QUESTION: What is the role of a prototype program in problem solving? <STOP>
REFERENCE ANSWER: To simulate the behaviour of portions of the desired software product. <STOP>
STUDENT ANSWER: High risk problems are address in the prototype program to make sure that the program is feasible. <STOP> A prototype may also be used to show a company that the software can be possibly programmed. <STOP>
Score (0-5):<|im_end|>
<|im_start|>assistant



In [6]:
# 5) Deterministic (greedy) batched generation with incremental saving so that an interrupted run can resume
tag=MODEL.split('/')[-1]; ckpt=f'{OUT}/llm_scores_v21_{tag}.csv'
done=pd.read_csv(ckpt) if os.path.exists(ckpt) else pd.DataFrame(columns=['id','llm_raw','llm_score'])
start=len(done); BS=8; t0=time.time(); rows=done.to_dict('records')
def parse(s):
    m=re.search(r'\d+(?:\.\d+)?', s); v=float(m.group()) if m else np.nan
    return float(np.clip(round(v*2)/2, 0, 5)) if not np.isnan(v) else np.nan
for i in range(start, len(df), BS):
    batch=prompts[i:i+BS]; enc=tok(batch, return_tensors='pt', padding=True).to(model.device)
    with torch.no_grad(): out=model.generate(**enc, max_new_tokens=6, do_sample=False, pad_token_id=tok.eos_token_id)
    texts=tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    for j,t in enumerate(texts): rows.append({'id': df.id.iloc[i+j], 'llm_raw': t.strip(), 'llm_score': parse(t)})
    if (i//BS)%10==0 or i+BS>=len(df):
        pd.DataFrame(rows).to_csv(ckpt, index=False); print(f'{len(rows)}/{len(df)} ({time.time()-t0:.0f}s)', flush=True)
res=pd.DataFrame(rows); print('unparsed:', res.llm_score.isna().sum()); print(res.llm_score.describe())

8/2442 (4s)
88/2442 (23s)
168/2442 (40s)
248/2442 (60s)
328/2442 (82s)
408/2442 (104s)
488/2442 (122s)
568/2442 (142s)
648/2442 (164s)
728/2442 (187s)
808/2442 (208s)
888/2442 (229s)
968/2442 (249s)
1048/2442 (270s)
1128/2442 (289s)
1208/2442 (310s)
1288/2442 (331s)
1368/2442 (353s)
1448/2442 (379s)
1528/2442 (401s)
1608/2442 (421s)
1688/2442 (440s)
1768/2442 (459s)
1848/2442 (478s)
1928/2442 (498s)
2008/2442 (519s)
2088/2442 (539s)
2168/2442 (561s)
2248/2442 (583s)
2328/2442 (603s)
2408/2442 (623s)
2442/2442 (635s)
unparsed: 0
count    2442.000000
mean        3.445127
std         0.978915
min         0.000000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: llm_score, dtype: float64


In [7]:
# 6) Preliminary evaluation of the LLM channel alone (reference = mean of the graders) and archive
from scipy.stats import pearsonr, spearmanr
m=df.merge(res, on='id'); p=m.llm_score.fillna(m.llm_score.median()).to_numpy(); r=m.score_avg.to_numpy(); e=p-r
summary={'model':MODEL,'n':int(len(m)),'unparsed':int(res.llm_score.isna().sum()),'data_hash':sha,'rmse':float(np.sqrt(np.mean(e**2))),'mae':float(np.abs(e).mean()),
         'pearson':float(pearsonr(p,r)[0]),'spearman':float(spearmanr(p,r)[0]),'acc_within_0.5':float((np.abs(e)<=0.5).mean()),'acc_within_1.0':float((np.abs(e)<=1).mean()),
         'system_prompt':SYSTEM,'decoding':'greedy, max_new_tokens=6, 4-bit NF4','seconds':time.time()-t0}
json.dump(summary, open(f'{OUT}/llm_summary_v21_{tag}.json','w'), indent=1); print(json.dumps(summary, indent=1))
!cd {OUT} && zip -q -r exp3_llm_v21_{tag}.zip llm_scores_v21_{tag}.csv llm_summary_v21_{tag}.json && ls -la {OUT}

{
 "model": "Qwen/Qwen2.5-7B-Instruct",
 "n": 2442,
 "unparsed": 0,
 "data_hash": "86b6af7101c1b3de",
 "rmse": 1.087641256700022,
 "mae": 0.9091420966420967,
 "pearson": 0.7199077227071919,
 "spearman": 0.6826627107813174,
 "acc_within_0.5": 0.3325143325143325,
 "acc_within_1.0": 0.8542178542178542,
 "system_prompt": "You are an experienced computer-science instructor grading short-answer questions. Grade the STUDENT ANSWER against the REFERENCE ANSWER on a 0-5 scale (half points allowed): 5 = fully correct and complete; 4 = correct with a minor omission; 3 = partially correct; 2 = mostly incorrect but shows some understanding; 1 = largely incorrect; 0 = wrong, empty or irrelevant. Judge meaning, not wording. Output ONLY the number.",
 "decoding": "greedy, max_new_tokens=6, 4-bit NF4",
 "seconds": 667.6395492553711
}
total 59
drwx------  2 root root  4096 Sep 23 17:03 .
drwx------ 16 root root  4096 Sep 23 16:39 ..
-rw-------  1 root root  6974 Sep 23 17:03 exp3_llm_v21_Qwen2.5-7B-Inst

## README — v21
- This notebook produces the LLM channel of Algorithm 3 only; the fusion (mean of TM and LLM) and the final evaluation with five 70/30 splits are performed on the output file in `FedEduAI_Validation_v13.ipynb`.
- Saving is incremental: if the session is interrupted, re-running cells 1–5 resumes from the last saved batch.
- If an out-of-memory error occurs in cell 3, set `MODEL` to `Qwen/Qwen2.5-3B-Instruct` and re-run; the model name is part of the output file names.
- The output archive is `exp3_llm_v21_<model>.zip` in `iudb/results/exp3/`.
